# Week 7 — Content Action Playbook

**Author:** Zain-ul-Abdeen
**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring
**Assignment:** ML-10

## 1. Ranked Actions + Reason Codes

We transition our model scores (Missed Clicks = Expected CTR - Actual CTR * Impressions) into concrete content actions. 

**Archetype -> Action Mapping:**
- **High Impressions, Top 3 Rank, Severe CTR Underperformance** 
  -> *Action:* `REVIEW_TITLE_AND_META`. 
  -> *Reason Code:* `CTR_GAP_HIGH_VISIBILITY`
- **Mid Impressions, Rank 4-10, Moderate CTR Underperformance** 
  -> *Action:* `EVALUATE_SERP_FEATURES`. 
  -> *Reason Code:* `CTR_GAP_MID_VISIBILITY` (Investigate if Featured Snippets or local packs are suppressing blue-link clicks).
- **High Impressions, Rank 1-2, Near-Zero CTR** 
  -> *Action:* `DO_NOTHING_ZERO_CLICK_SERP`. 
  -> *Reason Code:* `ZERO_CLICK_ANOMALY` (Likely a calculator, time zone, or pure definition query handled by Google natively).

**The Decay/Refresh Insight:**
If a page has slowly lost CTR over the last 90 days despite holding its position, the title may be suffering from staleness (e.g., still having "2025 Guide" in the title when it's now 2026). 

In [ ]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

con = duckdb.connect()

# Authenticate with Hugging Face (Paste your token in Colab)
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Build the final actionable queue for the playbook (using our baseline rule logic for simplicity)
queue_query = f"""
    WITH page_stats AS (
        SELECT content_hash_id,
               SUM(gsc_clicks) as clicks,
               SUM(gsc_impressions) as impressions,
               AVG(gsc_avg_position) as avg_pos,
               SUM(gsc_clicks)/SUM(gsc_impressions) as actual_ctr
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) > 500
    ),
    scored AS (
        SELECT 
            content_hash_id, clicks, impressions, avg_pos, actual_ctr,
            CASE 
                WHEN avg_pos <= 3 THEN 0.20
                WHEN avg_pos <= 10 THEN 0.05
                ELSE 0.01
            END as expected_ctr
        FROM page_stats
    )
    SELECT 
        content_hash_id,
        impressions, avg_pos, actual_ctr,
        ROUND((expected_ctr - actual_ctr) * impressions, 0) as missed_clicks,
        CASE
            WHEN avg_pos <= 3 AND actual_ctr < 0.02 THEN 'DO_NOTHING_ZERO_CLICK_SERP'
            WHEN avg_pos <= 3 AND actual_ctr < 0.20 THEN 'REVIEW_TITLE_AND_META'
            WHEN avg_pos <= 10 AND actual_ctr < 0.05 THEN 'EVALUATE_SERP_FEATURES'
            ELSE 'NO_ACTION'
        END as action_label,
        CASE
            WHEN avg_pos <= 3 AND actual_ctr < 0.02 THEN 'ZERO_CLICK_ANOMALY'
            WHEN avg_pos <= 3 AND actual_ctr < 0.20 THEN 'CTR_GAP_HIGH_VISIBILITY'
            WHEN avg_pos <= 10 AND actual_ctr < 0.05 THEN 'CTR_GAP_MID_VISIBILITY'
            ELSE 'NONE'
        END as reason_code
    FROM scored
    WHERE expected_ctr > actual_ctr
    ORDER BY missed_clicks DESC
    LIMIT 500
"""
playbook_queue = con.sql(queue_query).df()
print("Top 5 Playbook Actions:")
display(playbook_queue.head())

## 2. Intended Use and Limits

**Intended Use:**
This playbook serves as a **triage and decision-support tool** for SEO and content teams. Instead of manually scrolling through Search Console for underperforming pages, teams start their week looking at the top 10 `REVIEW_TITLE_AND_META` flags, ordered strictly by missed click volume.

**Limits (Where the model fails):**
- **SERP Blindness:** The model sees average position, not pixel depth. If a page ranks #1 but is pushed below the fold by Google Ads, Shopping Carousels, and AI Overviews, the model will flag it as an underperformer, even though the organic CTR is perfectly normal for that squeezed real estate.
- **Branded vs Non-Branded:** The model currently groups all impressions together. A competitor's branded term where we rank #2 will naturally have a <1% CTR. The model will flag it as an opportunity, but rewriting the title won't steal a navigational click meant for the competitor.

## 3. Human Review + The No-Go List

**Human Review:**
Every `REVIEW_TITLE_AND_META` action requires an editor to actually look at the live Google SERP before rewriting the title. The editor must confirm that our current title is objectively worse or less compelling than the surrounding competitors.

**The NO-GO List (What should NOT be automated):**
- **Automated Title Deployment:** We should **never** allow an AI model to automatically rewrite and publish the `<title>` tags based on these flags. Changing a title tag can inadvertently drop the page's ranking entirely. The risk of ruining a top-3 ranking far outweighs the reward of a slight CTR bump.
- **Zero-Click Interventions:** Pages flagged as `ZERO_CLICK_ANOMALY` should be excluded from review queues entirely so we don't waste editor time fighting Google's native answers (e.g. "What is my IP?").

## 4. Monitoring / Retrain Triggers

**Cost/Value Thinking:** 
Running this model and queue generation costs pennies in compute. The real cost is editor time. If we flag 100 pages a week, that's ~10 hours of editor review time. To maximize ROI, we only surface the Top 20 flags where `missed_clicks > 500`.

**Monitoring Rules:**
- Track the **Action Completion Rate**: What percentage of flagged pages actually receive a title rewrite? If it drops below 10%, the model is generating too many false positives (e.g., flagging un-winnable SERPs).

**Retrain Triggers:**
1. **Major Google Core Updates:** If Google significantly changes SERP layouts (e.g., rolling out AI Overviews to 100% of queries), our Expected CTR baselines (20% for rank 1) will become invalid. The model must be retrained on post-update data.
2. **Feature Drift:** If the median CTR of the `ZERO_CLICK_ANOMALY` bucket suddenly changes, it triggers a manual audit of the model's threshold logic.

## 5. Exports for the Paper

We will export the queue to `work/outputs/action_playbook_queue.csv` and generate a single visualization of the Expected vs Actual CTR decay to `work/figures/ctr_decay.png`.

In [ ]:
import os
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export the queue (this file is gitignored)
playbook_queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print("Exported queue to work/outputs/action_playbook_queue.csv")

# Generate a Figure for the research paper (reusing baseline expected CTR logic vs actual)
decay_query = f"""
    WITH pos_group AS (
        SELECT 
            FLOOR(gsc_avg_position) as pos_bucket,
            SUM(gsc_clicks)/SUM(gsc_impressions) * 100 as actual_ctr
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) > 1000 AND pos_bucket <= 15
    )
    SELECT * FROM pos_group ORDER BY pos_bucket
"""
decay_df = con.sql(decay_query).df()

plt.figure(figsize=(10, 6))
plt.plot(decay_df['pos_bucket'], decay_df['actual_ctr'], marker='o', linestyle='-', color='#1f77b4', label='Actual CTR (Mar 2026)')

# Overlay the baseline threshold expectation
expected_y = [20 if p<=3 else 5 if p<=10 else 1 for p in decay_df['pos_bucket']]
plt.plot(decay_df['pos_bucket'], expected_y, marker='', linestyle='--', color='#d62728', label='Baseline Threshold')

plt.title('Organic CTR Decay by Position (Lane 4 Opportunity Gap)')
plt.xlabel('Average Search Position')
plt.ylabel('Click-Through Rate (%)')
plt.xticks(np.arange(1, 16, 1))
plt.grid(alpha=0.3)
plt.legend()

plt.savefig('work/figures/ctr_decay.png', dpi=300, bbox_inches='tight')
print("Exported figure to work/figures/ctr_decay.png")
plt.show()

## 6. Self-Check

| Check | Answer |
|---|---|
| **Ranked actions & reason codes mapped?** | Yes. Mapped to `REVIEW_TITLE_AND_META`, `EVALUATE_SERP_FEATURES`, and `DO_NOTHING_ZERO_CLICK_SERP`. |
| **Intended use and limits explained?** | Yes, covered SERP blindness and branded vs non-branded limitations. |
| **Human review & NO-GO list stated?** | Yes. Explicitly forbade automated title deployment. |
| **Monitoring/retrain triggers and cost/value?** | Yes. Action completion rate as a metric, and Core Updates as a retrain trigger. |
| **Exports generated for the paper?** | Yes, the queue CSV to `work/outputs/` and a CTR decay plot to `work/figures/`. |

## 7. 5-Minute Demo Showcase Outline

**Target Duration:** 5 Minutes | **Audience:** ML Engineers, SEO Leads, Product Stakeholders

### Minute 1: The Core Question & Motivation
- **The Problem:** FlyRank automates organic growth for client portfolios, but standard SEO workflows struggle to pinpoint high-value content fixes. Raw CTR is misleading (5% CTR at Rank 1 is broken; 5% CTR at Rank 15 is great).
- **The Question:** *Which pages exhibit the highest CTR Opportunity Gap—underperforming expected position-based CTR weighted by search volume?*

### Minute 2: Data & Methodology
- **The Dataset:** 79M-row production warehouse of hashed GSC metrics + GA4 engagement data (`month=2026-03` slice, minimum 500 impressions threshold).
- **The Modeling Approach:** Started with a transparent 3-bucket heuristic baseline (Rank 1-3 = 20% CTR, 4-10 = 5%, 11+ = 1%), then trained a Random Forest Regressor on `avg_pos`, `sessions`, and `content_intent` to model continuous CTR decay.

### Minute 3: The Key Visual & Chart
- **The Chart (`work/figures/ctr_decay.png`):** Shows actual organic CTR decay vs. baseline expectation by search position.
- **The Insight:** Visualizes where the gap is largest (Position 1-3) and highlights why rank alone does not explain click share.

### Minute 4: Honest Validation & The Hard Truth
- **Validation Audit:** Evaluated models using `GroupShuffleSplit` across `client_hash_id` (preventing domain-level CTR memorization).
- **The Honest Result:** Random Forest achieves superior MAE over the rule-based baseline on unseen clients, but R² drops from naive splits to grouped splits—proving the necessity of client-level holdout validation.
- **The Limit (SERP Blindness):** The model cannot see AI Overviews, ads, or zero-click elements that compress real estate.

### Minute 5: Actionable Recommendations
- **The Playbook:** Ranked triage queue outputting `REVIEW_TITLE_AND_META`, `EVALUATE_SERP_FEATURES`, and `DO_NOTHING_ZERO_CLICK_SERP`.
- **The Golden Rule:** *Never automate title tag publication.* Keep humans in the loop to preserve high-ranking positions while improving click efficiency.


## 8. Two Shareable Cuts of This Work

### Cut 1: Methodology Social Post (LinkedIn / X)
```text
Most SEO models make a fatal assumption: evaluating models on random splits of page URLs.

During the FlyRank ML Internship, I built an anomaly detection pipeline on 79M rows of search performance data to identify "CTR Opportunity Gaps"—pages with high search rankings but sub-par click-through rates.

Here's what our validation audit revealed:
1. Naive random splits look great on paper because the model memorizes client-specific domain patterns.
2. Moving to a Grouped Client Split (GroupShuffleSplit) exposed the true generalization error on brand new client portfolios.
3. Incorporating GA4 user session context and continuous non-linear decay curves beat fixed threshold rules by a wide margin.

Key takeaway: In search optimization ML, leakage is subtle and domain structure matters. Never automate title deployment without human-in-the-loop verification!

Full paper & reproducible notebooks: https://zain-ul-abdeen-773.github.io/flyrank-ml-internship/
```

---

### Cut 2: Employer-Facing Summary (Resume / Interview 3-Sentence Elevator Pitch)
> *"Trained and audited a Random Forest regression pipeline on a 79-million-row production search warehouse to quantify CTR opportunity gaps across client content portfolios. Validated generalization using a client-grouped holdout design (GroupShuffleSplit) and proved that non-linear position decay curves with GA4 session features significantly outperform rule-based baselines in MAE. Synthesized the findings into a deployed research paper and an automated editorial triage playbook that surfaces high-value title optimization candidates while guarding against SERP blindness."*